FRTB-SA Capital Calculation Framework
1. Trade Book Creation
Create the trading portfolio containing positions, instruments, quantities, prices, and trade attributes.

2. Trade Book Validation
Validate trade completeness, data quality, missing values, and position consistency.

3. Market Data Preparation
Prepare market data inputs including interest rates, equity prices, FX rates, commodity prices, credit spreads, and volatility surfaces.

4. Risk Factor Mapping
Map each trade to its relevant regulatory risk factors and risk classes.

5. Delta Sensitivity Calculation
Calculate first-order sensitivities measuring the impact of small changes in market risk factors.

6. Vega Sensitivity Calculation
Calculate volatility sensitivities measuring the impact of changes in implied volatility.

7. Curvature Risk Calculation
Calculate non-linear option risk resulting from large market shocks and changes in Delta.

8. Default Risk Charge (DRC)
Measure jump-to-default risk arising from the potential default of issuers and counterparties.

9. Residual Risk Add-On (RRAO)
Calculate additional capital for exotic and residual risks not adequately captured by other components.

10. Sensitivities-Based Method (SBM) Aggregation
Aggregate Delta, Vega, and Curvature risks using prescribed risk weights and correlation matrices. 'Delta Risk > Vega Risk > Curvature Risk'

11. Total FRTB-SA Capital Calculation
FRTB-SA Capital = SBM + DRC + RRAO
SBM = Delta Capital + Vega Capital + Curvature Capital

12. Final Risk Reporting
Generate regulatory capital reports, capital breakdowns by risk class, risk factor summaries, reconciliation results, and management dashboards.

Overall Process Flow
Trading Book ↓ Trade Validation ↓ Market Data Preparation ↓ Risk Factor Mapping ↓ Delta Calculation ↓ Vega Calculation ↓ Curvature Calculation ↓ DRC Calculation ↓ RRAO Calculation ↓ SBM Aggregation ↓ Total FRTB-SA Capital ↓ Final Risk Reporting

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import os

# Set project folder
project_path = "/Users/krishnakumarjena/Downloads/Projects/Market Risk/FRTB SA"

os.makedirs(project_path, exist_ok=True)
print("Project folder created successfully.")

Project folder created successfully.


Step 1 – Trade Book Creation

In [2]:
# Create sample trading book

trades = [
    ["TR001", "Apple Stock", "Equity", "Cash Equity", "Long", 500000, 500000, "USD", "Apple", None, None],
    ["TR002", "Microsoft Stock", "Equity", "Cash Equity", "Long", 300000, 300000, "USD", "Microsoft", None, None],
    ["TR003", "US Treasury Bond", "Interest Rate", "Government Bond", "Long", 1000000, 980000, "USD", "US Government", "2030-12-31", None],
    ["TR004", "Corporate Bond A", "Credit", "Corporate Bond", "Long", 800000, 780000, "USD", "Company A", "2029-06-30", None],
    ["TR005", "EUR/USD Forward", "FX", "FX Forward", "Long", 500000, 500000, "USD", None, "2027-06-30", None],
    ["TR006", "Gold Futures", "Commodity", "Futures", "Long", 400000, 400000, "USD", None, "2027-03-31", None],
    ["TR007", "Apple Call Option", "Equity", "Option", "Long", 200000, 25000, "USD", "Apple", "2027-01-15", "Call"],
    ["TR008", "Microsoft Put Option", "Equity", "Option", "Long", 150000, 18000, "USD", "Microsoft", "2027-02-15", "Put"],
    ["TR009", "Interest Rate Swap", "Interest Rate", "IRS", "Long", 2000000, 35000, "USD", "Bank A", "2030-12-31", None],
    ["TR010", "USD/INR Forward", "FX", "FX Forward", "Long", 600000, 600000, "USD", None, "2027-09-30", None],
    ["TR011", "Silver Futures", "Commodity", "Futures", "Short", 300000, 300000, "USD", None, "2027-04-30", None],
    ["TR012", "Corporate Bond B", "Credit", "Corporate Bond", "Long", 900000, 875000, "USD", "Company B", "2031-03-31", None]
]

columns = ["Trade_ID", "Instrument", "Asset_Class", "Product_Type", "Position", "Notional", "Market_Value", "Currency", "Issuer", "Maturity", "Option_Type"]

trade_book = pd.DataFrame(trades, columns=columns)

# Convert maturity into date format
trade_book["Maturity"] = pd.to_datetime(trade_book["Maturity"], errors="coerce")

print(trade_book)

   Trade_ID            Instrument    Asset_Class     Product_Type Position  \
0     TR001           Apple Stock         Equity      Cash Equity     Long   
1     TR002       Microsoft Stock         Equity      Cash Equity     Long   
2     TR003      US Treasury Bond  Interest Rate  Government Bond     Long   
3     TR004      Corporate Bond A         Credit   Corporate Bond     Long   
4     TR005       EUR/USD Forward             FX       FX Forward     Long   
5     TR006          Gold Futures      Commodity          Futures     Long   
6     TR007     Apple Call Option         Equity           Option     Long   
7     TR008  Microsoft Put Option         Equity           Option     Long   
8     TR009    Interest Rate Swap  Interest Rate              IRS     Long   
9     TR010       USD/INR Forward             FX       FX Forward     Long   
10    TR011        Silver Futures      Commodity          Futures    Short   
11    TR012      Corporate Bond B         Credit   Corporate Bon

In [3]:
print("Total Trades:", len(trade_book))

print("\nAsset Class Distribution:")
print(trade_book["Asset_Class"].value_counts())

print("\nTotal Notional:", trade_book["Notional"].sum())

print("\nTotal Market Value:", trade_book["Market_Value"].sum())

Total Trades: 12

Asset Class Distribution:
Asset_Class
Equity           4
Interest Rate    2
Credit           2
FX               2
Commodity        2
Name: count, dtype: int64

Total Notional: 7650000

Total Market Value: 5313000


In [4]:
trade_book.to_csv(os.path.join(project_path, "trade_book.csv"), index=False)

print("Trade book exported successfully.")

Trade book exported successfully.


Step 2 – Trade Book Validation

In [5]:
# Check duplicate Trade IDs
duplicates = trade_book[trade_book["Trade_ID"].duplicated()]
print("Duplicate Trades:", len(duplicates))


# Check missing values
print("\nMissing Values:")
print(trade_book.isnull().sum())


# Validate asset classes
valid_asset_classes = ["Equity", "Interest Rate", "Credit", "FX", "Commodity"]
invalid_assets = trade_book[~trade_book["Asset_Class"].isin(valid_asset_classes)]
print("\nInvalid Asset Classes:", len(invalid_assets))


# Validate positions
valid_positions = ["Long", "Short"]
invalid_positions = trade_book[~trade_book["Position"].isin(valid_positions)]
print("Invalid Positions:", len(invalid_positions))


# Validate product types
valid_products = ["Cash Equity", "Government Bond", "Corporate Bond", "FX Forward", "Futures", "Option", "IRS"]
invalid_products = trade_book[~trade_book["Product_Type"].isin(valid_products)]
print("Invalid Product Types:", len(invalid_products))


# Validate notional and market values
invalid_values = trade_book[(trade_book["Notional"] <= 0) | (trade_book["Market_Value"] <= 0)]
print("Invalid Value Records:", len(invalid_values))


# Prepare validation summary
validation_summary = pd.DataFrame({
    "Validation Check": [
        "Duplicate Trade IDs",
        "Missing Values",
        "Invalid Asset Classes",
        "Invalid Positions",
        "Invalid Product Types",
        "Invalid Notional / Market Values"
    ],
    "Records Found": [
        len(duplicates),
        trade_book.isnull().sum().sum(),
        len(invalid_assets),
        len(invalid_positions),
        len(invalid_products),
        len(invalid_values)
    ]
})

print("\nValidation Summary:")
print(validation_summary)


# Export validation results
validation_summary.to_csv(os.path.join(project_path, "trade_validation_summary.csv"), index=False)

print("\nValidation summary exported successfully.")

Duplicate Trades: 0

Missing Values:
Trade_ID         0
Instrument       0
Asset_Class      0
Product_Type     0
Position         0
Notional         0
Market_Value     0
Currency         0
Issuer           4
Maturity         2
Option_Type     10
dtype: int64

Invalid Asset Classes: 0
Invalid Positions: 0
Invalid Product Types: 0
Invalid Value Records: 0

Validation Summary:
                   Validation Check  Records Found
0               Duplicate Trade IDs              0
1                    Missing Values             16
2             Invalid Asset Classes              0
3                 Invalid Positions              0
4             Invalid Product Types              0
5  Invalid Notional / Market Values              0

Validation summary exported successfully.


Step 3 – Market Data Preparation

In [6]:
pip install yfinance


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import yfinance as yf

# Define market data instruments
tickers = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "GC=F": "Gold",
    "SI=F": "Silver",
    "EURUSD=X": "EUR/USD",
    "USDINR=X": "USD/INR"
}

# Download historical market prices
market_prices = pd.DataFrame()

for ticker, name in tickers.items():
    data = yf.download(ticker, start="2022-01-01", end="2026-01-01", auto_adjust=True, progress=False)

    if not data.empty:
        market_prices[name] = data["Close"].squeeze()

# Remove missing observations
market_prices = market_prices.dropna(how="all")

# Check historical market data
print("Market Data Shape:", market_prices.shape)
print("\nMissing Values:")
print(market_prices.isnull().sum())
print("\nDate Range:", market_prices.index.min(), "to", market_prices.index.max())


# Create synthetic interest rate curve
interest_rates = pd.DataFrame({
    "Currency": ["USD", "USD", "USD", "USD", "USD"],
    "Tenor": ["1Y", "2Y", "5Y", "10Y", "30Y"],
    "Interest_Rate": [4.25, 4.10, 4.00, 4.20, 4.50]
})


# Create synthetic credit spreads
credit_spreads = pd.DataFrame({
    "Issuer": ["Company A", "Company B"],
    "Rating": ["BBB", "A"],
    "Credit_Spread": [1.80, 1.20]
})


# Create synthetic implied volatility data
implied_volatility = pd.DataFrame({
    "Instrument": ["Apple Call Option", "Microsoft Put Option"],
    "Underlying": ["Apple", "Microsoft"],
    "Implied_Volatility": [25.0, 28.0]
})


# Export market data
market_prices.to_csv(os.path.join(project_path, "historical_market_prices.csv"))
interest_rates.to_csv(os.path.join(project_path, "interest_rate_curve.csv"), index=False)
credit_spreads.to_csv(os.path.join(project_path, "credit_spreads.csv"), index=False)
implied_volatility.to_csv(os.path.join(project_path, "implied_volatility.csv"), index=False)

print("\nMarket data preparation completed.")
print("All market data files exported successfully.")

Market Data Shape: (1003, 6)

Missing Values:
Apple        0
Microsoft    0
Gold         0
Silver       0
EUR/USD      1
USD/INR      1
dtype: int64

Date Range: 2022-01-03 00:00:00 to 2025-12-31 00:00:00

Market data preparation completed.
All market data files exported successfully.


Step 4 – Risk Factor Mapping

In [8]:
# Map trades to their market risk factors

risk_mapping = [
    ["TR001", "Equity", "Apple Share Price", "Apple", "Delta"],
    ["TR002", "Equity", "Microsoft Share Price", "Microsoft", "Delta"],
    ["TR003", "GIRR", "USD Interest Rate", "USD", "Delta"],
    ["TR004", "GIRR", "USD Interest Rate", "USD", "Delta"],
    ["TR004", "CSR", "Company A Credit Spread", "Company A", "Delta"],
    ["TR005", "FX", "EUR/USD Exchange Rate", "EUR/USD", "Delta"],
    ["TR006", "Commodity", "Gold Price", "Gold", "Delta"],
    ["TR007", "Equity", "Apple Share Price", "Apple", "Delta"],
    ["TR007", "Equity", "Apple Implied Volatility", "Apple", "Vega"],
    ["TR008", "Equity", "Microsoft Share Price", "Microsoft", "Delta"],
    ["TR008", "Equity", "Microsoft Implied Volatility", "Microsoft", "Vega"],
    ["TR009", "GIRR", "USD Interest Rate", "USD", "Delta"],
    ["TR010", "FX", "USD/INR Exchange Rate", "USD/INR", "Delta"],
    ["TR011", "Commodity", "Silver Price", "Silver", "Delta"],
    ["TR012", "GIRR", "USD Interest Rate", "USD", "Delta"],
    ["TR012", "CSR", "Company B Credit Spread", "Company B", "Delta"]
]

mapping_columns = ["Trade_ID", "Risk_Class", "Risk_Factor", "Risk_Bucket", "Sensitivity_Type"]

risk_factor_mapping = pd.DataFrame(risk_mapping, columns=mapping_columns)

print(risk_factor_mapping)

   Trade_ID Risk_Class                   Risk_Factor Risk_Bucket  \
0     TR001     Equity             Apple Share Price       Apple   
1     TR002     Equity         Microsoft Share Price   Microsoft   
2     TR003       GIRR             USD Interest Rate         USD   
3     TR004       GIRR             USD Interest Rate         USD   
4     TR004        CSR       Company A Credit Spread   Company A   
5     TR005         FX         EUR/USD Exchange Rate     EUR/USD   
6     TR006  Commodity                    Gold Price        Gold   
7     TR007     Equity             Apple Share Price       Apple   
8     TR007     Equity      Apple Implied Volatility       Apple   
9     TR008     Equity         Microsoft Share Price   Microsoft   
10    TR008     Equity  Microsoft Implied Volatility   Microsoft   
11    TR009       GIRR             USD Interest Rate         USD   
12    TR010         FX         USD/INR Exchange Rate     USD/INR   
13    TR011  Commodity                  Silver P

In [9]:
print("Risk Class Distribution:")
print(risk_factor_mapping["Risk_Class"].value_counts())

print("\nSensitivity Type Distribution:")
print(risk_factor_mapping["Sensitivity_Type"].value_counts())

Risk Class Distribution:
Risk_Class
Equity       6
GIRR         4
CSR          2
FX           2
Commodity    2
Name: count, dtype: int64

Sensitivity Type Distribution:
Sensitivity_Type
Delta    14
Vega      2
Name: count, dtype: int64


In [10]:
mapped_trades = risk_factor_mapping["Trade_ID"].unique()

unmapped_trades = trade_book[~trade_book["Trade_ID"].isin(mapped_trades)]

print("Unmapped Trades:", len(unmapped_trades))
print(unmapped_trades)

Unmapped Trades: 0
Empty DataFrame
Columns: [Trade_ID, Instrument, Asset_Class, Product_Type, Position, Notional, Market_Value, Currency, Issuer, Maturity, Option_Type]
Index: []


In [11]:
risk_factor_mapping.to_csv(os.path.join(project_path, "risk_factor_mapping.csv"), index=False)

print("Risk factor mapping exported successfully.")

Risk factor mapping exported successfully.


Step 5 – Delta Sensitivity Calculation

In [12]:
# Define option Delta assumptions
option_deltas = {
    "TR007": 0.60,
    "TR008": -0.40
}

# Define PV01 assumptions for interest rate and credit spread risk
pv01_values = {
    ("TR003", "GIRR"): -850,
    ("TR004", "GIRR"): -600,
    ("TR004", "CSR"): -600,
    ("TR009", "GIRR"): -1200,
    ("TR012", "GIRR"): -700,
    ("TR012", "CSR"): -700
}

# Join trade details with risk factor mapping
delta_data = risk_factor_mapping[risk_factor_mapping["Sensitivity_Type"] == "Delta"].merge(trade_book, on="Trade_ID")

# Calculate signed Delta sensitivity
delta_sensitivities = []

for _, row in delta_data.iterrows():

    position_sign = 1 if row["Position"] == "Long" else -1

    if row["Risk_Class"] == "Equity" and row["Product_Type"] == "Cash Equity":
        sensitivity = row["Market_Value"] * 0.01 * position_sign

    elif row["Product_Type"] == "Option":
        sensitivity = row["Notional"] * option_deltas[row["Trade_ID"]] * 0.01 * position_sign

    elif row["Risk_Class"] == "FX":
        sensitivity = row["Notional"] * 0.01 * position_sign

    elif row["Risk_Class"] == "Commodity":
        sensitivity = row["Notional"] * 0.01 * position_sign

    elif row["Risk_Class"] in ["GIRR", "CSR"]:
        sensitivity = pv01_values[(row["Trade_ID"], row["Risk_Class"])] * position_sign

    else:
        sensitivity = np.nan

    delta_sensitivities.append(sensitivity)

delta_data["Delta_Sensitivity"] = delta_sensitivities

# Display trade-level Delta sensitivities
delta_report = delta_data[["Trade_ID", "Instrument", "Risk_Class", "Risk_Factor", "Delta_Sensitivity"]]

print(delta_report)

# Export Delta sensitivity report
delta_report.to_csv(os.path.join(project_path, "delta_sensitivity_report.csv"), index=False)

print("\nDelta sensitivity report exported successfully.")

   Trade_ID            Instrument Risk_Class              Risk_Factor  \
0     TR001           Apple Stock     Equity        Apple Share Price   
1     TR002       Microsoft Stock     Equity    Microsoft Share Price   
2     TR003      US Treasury Bond       GIRR        USD Interest Rate   
3     TR004      Corporate Bond A       GIRR        USD Interest Rate   
4     TR004      Corporate Bond A        CSR  Company A Credit Spread   
5     TR005       EUR/USD Forward         FX    EUR/USD Exchange Rate   
6     TR006          Gold Futures  Commodity               Gold Price   
7     TR007     Apple Call Option     Equity        Apple Share Price   
8     TR008  Microsoft Put Option     Equity    Microsoft Share Price   
9     TR009    Interest Rate Swap       GIRR        USD Interest Rate   
10    TR010       USD/INR Forward         FX    USD/INR Exchange Rate   
11    TR011        Silver Futures  Commodity             Silver Price   
12    TR012      Corporate Bond B       GIRR       

Step 6 – Vega Sensitivity Calculation

In [13]:
# Define assumed Vega values
option_vega = {
    "TR007": 1200,
    "TR008": 900
}

# Select only Vega risk factor records
vega_data = risk_factor_mapping[risk_factor_mapping["Sensitivity_Type"] == "Vega"].copy()

# Join with trade book
vega_data = vega_data.merge(trade_book, on="Trade_ID")

# Join with implied volatility data
vega_data = vega_data.merge(implied_volatility, on="Instrument", how="left")

# Calculate signed Vega sensitivity
vega_data["Vega_Sensitivity"] = vega_data.apply(
    lambda row: option_vega[row["Trade_ID"]] * (1 if row["Position"] == "Long" else -1),
    axis=1
)

# Display Vega sensitivity report
vega_report = vega_data[["Trade_ID", "Instrument", "Risk_Class", "Risk_Factor", "Implied_Volatility", "Vega_Sensitivity"]]

print(vega_report)

# Export Vega sensitivity report
vega_report.to_csv(os.path.join(project_path, "vega_sensitivity_report.csv"), index=False)

print("\nVega sensitivity report exported successfully.")

  Trade_ID            Instrument Risk_Class                   Risk_Factor  \
0    TR007     Apple Call Option     Equity      Apple Implied Volatility   
1    TR008  Microsoft Put Option     Equity  Microsoft Implied Volatility   

   Implied_Volatility  Vega_Sensitivity  
0                25.0              1200  
1                28.0               900  

Vega sensitivity report exported successfully.


Step 7 – Curvature Risk Calculation

In [14]:
# Define assumed Gamma values for options
option_gamma = {
    "TR007": 0.020,
    "TR008": 0.015
}

# Select option trades and their underlying equity risk factors
curvature_data = risk_factor_mapping[
    (risk_factor_mapping["Sensitivity_Type"] == "Delta") &
    (risk_factor_mapping["Trade_ID"].isin(option_gamma))
].merge(trade_book, on="Trade_ID")

# Get the latest underlying market prices
curvature_data["Underlying_Price"] = curvature_data["Risk_Bucket"].apply(
    lambda x: market_prices[x].dropna().iloc[-1]
)

# Calculate the approximate curvature effect
curvature_data["Gamma"] = curvature_data["Trade_ID"].map(option_gamma)
curvature_data["Position_Sign"] = curvature_data["Position"].map({"Long": 1, "Short": -1})
curvature_data["Underlying_Quantity"] = curvature_data["Notional"] / curvature_data["Underlying_Price"]

# Apply an illustrative 15% market shock
curvature_data["Shock_Size"] = curvature_data["Underlying_Price"] * 0.15

# Calculate the non-linear P&L component
curvature_data["Curvature_PnL"] = (
    0.5 * curvature_data["Gamma"] *
    curvature_data["Underlying_Quantity"] *
    curvature_data["Shock_Size"] ** 2 *
    curvature_data["Position_Sign"]
)

# Calculate the adverse curvature loss
curvature_data["Curvature_Charge"] = (-curvature_data["Curvature_PnL"]).clip(lower=0)

# Prepare the curvature risk report
curvature_report = curvature_data[
    ["Trade_ID", "Instrument", "Risk_Factor", "Underlying_Price",
     "Gamma", "Shock_Size", "Curvature_PnL", "Curvature_Charge"]
]

print(curvature_report)

# Export curvature report
curvature_report.to_csv(
    os.path.join(project_path, "curvature_risk_report.csv"),
    index=False
)

print("\nCurvature risk report exported successfully.")

  Trade_ID            Instrument            Risk_Factor  Underlying_Price  \
0    TR007     Apple Call Option      Apple Share Price        271.122009   
1    TR008  Microsoft Put Option  Microsoft Share Price        480.571167   

   Gamma  Shock_Size  Curvature_PnL  Curvature_Charge  
0  0.020   40.668301   12200.490417               0.0  
1  0.015   72.085675   12164.457664               0.0  

Curvature risk report exported successfully.


Step 8 – Default Risk Charge (DRC)

In [15]:
# Select corporate bond trades
drc_data = trade_book[
    trade_book["Product_Type"] == "Corporate Bond"
].copy()

# Join credit ratings
drc_data = drc_data.merge(credit_spreads, on="Issuer", how="left")

# Define illustrative recovery rates and risk weights
recovery_rates = {
    "Company A": 0.40,
    "Company B": 0.45
}

risk_weights = {
    "A": 0.005,
    "BBB": 0.005
}

# Calculate LGD, JTD and DRC
drc_data["Recovery_Rate"] = drc_data["Issuer"].map(recovery_rates)
drc_data["LGD"] = 1 - drc_data["Recovery_Rate"]
drc_data["JTD"] = drc_data["Notional"] * drc_data["LGD"]
drc_data["Risk_Weight"] = drc_data["Rating"].map(risk_weights)
drc_data["DRC"] = drc_data["JTD"] * drc_data["Risk_Weight"]

# Prepare DRC report
drc_report = drc_data[
    ["Trade_ID", "Instrument", "Issuer", "Rating", "Notional",
     "Recovery_Rate", "LGD", "JTD", "Risk_Weight", "DRC"]
]

print(drc_report)

# Calculate total DRC
total_drc = drc_report["DRC"].sum()

print("\nTotal Default Risk Charge:", round(total_drc, 2))

# Export DRC report
drc_report.to_csv(
    os.path.join(project_path, "default_risk_charge_report.csv"),
    index=False
)

print("\nDRC report exported successfully.")

  Trade_ID        Instrument     Issuer Rating  Notional  Recovery_Rate   LGD  \
0    TR004  Corporate Bond A  Company A    BBB    800000           0.40  0.60   
1    TR012  Corporate Bond B  Company B      A    900000           0.45  0.55   

        JTD  Risk_Weight     DRC  
0  480000.0        0.005  2400.0  
1  495000.0        0.005  2475.0  

Total Default Risk Charge: 4875.0

DRC report exported successfully.


Step 9 – Residual Risk Add-On (RRAO)

In [17]:
# Adding a hypothetical exotic option for RRAO demonstration
exotic_trade = pd.DataFrame([{
    "Trade_ID": "TR013",
    "Instrument": "Exotic Equity Option",
    "Asset_Class": "Equity",
    "Product_Type": "Exotic Option",
    "Position": "Long",
    "Notional": 500000,
    "Market_Value": 45000,
    "Currency": "USD",
    "Issuer": "Apple",
    "Maturity": pd.Timestamp("2027-12-31"),
    "Option_Type": "Exotic"
}])

trade_book = pd.concat([trade_book, exotic_trade], ignore_index=True)

# Export updated trade book
trade_book.to_csv(os.path.join(project_path, "trade_book.csv"), index=False)

print("Updated trade book:")
print(trade_book.tail())

Updated trade book:
   Trade_ID            Instrument Asset_Class    Product_Type Position  \
9     TR010       USD/INR Forward          FX      FX Forward     Long   
10    TR011        Silver Futures   Commodity         Futures    Short   
11    TR012      Corporate Bond B      Credit  Corporate Bond     Long   
12    TR013  Exotic Equity Option      Equity   Exotic Option     Long   
13    TR013  Exotic Equity Option      Equity   Exotic Option     Long   

    Notional  Market_Value Currency     Issuer   Maturity Option_Type  
9     600000        600000      USD        NaN 2027-09-30         NaN  
10    300000        300000      USD        NaN 2027-04-30         NaN  
11    900000        875000      USD  Company B 2031-03-31         NaN  
12    500000         45000      USD      Apple 2027-12-31      Exotic  
13    500000         45000      USD      Apple 2027-12-31      Exotic  


In [18]:
# Select trades identified as RRAO eligible
rrao_data = trade_book[
    trade_book["Product_Type"] == "Exotic Option"
].copy()

# Apply an illustrative add-on rate
rrao_rate = 0.01

rrao_data["RRAO_Rate"] = rrao_rate
rrao_data["RRAO"] = rrao_data["Notional"] * rrao_data["RRAO_Rate"]

# Prepare RRAO report
rrao_report = rrao_data[
    ["Trade_ID", "Instrument", "Product_Type", "Notional",
     "RRAO_Rate", "RRAO"]
]

print(rrao_report)

# Calculate total RRAO
total_rrao = rrao_report["RRAO"].sum()

print("\nTotal RRAO:", round(total_rrao, 2))

# Export RRAO report
rrao_report.to_csv(
    os.path.join(project_path, "rrao_report.csv"),
    index=False
)

print("\nRRAO report exported successfully.")

   Trade_ID            Instrument   Product_Type  Notional  RRAO_Rate    RRAO
12    TR013  Exotic Equity Option  Exotic Option    500000       0.01  5000.0
13    TR013  Exotic Equity Option  Exotic Option    500000       0.01  5000.0

Total RRAO: 10000.0

RRAO report exported successfully.


Step 10 – FRTB-SA Capital Calculation

In [19]:
# Summarise the risk components

capital_summary = pd.DataFrame([
    ["Delta", delta_report["Delta_Sensitivity"].abs().sum()],
    ["Vega", vega_report["Vega_Sensitivity"].abs().sum()],
    ["Curvature", curvature_report["Curvature_Charge"].sum()],
    ["DRC", drc_report["DRC"].sum()],
    ["RRAO", rrao_report["RRAO"].sum()]
], columns=["Risk_Component", "Calculated_Amount"])

print("FRTB-SA Risk Component Summary")
print(capital_summary)

# Export the summary
capital_summary.to_csv(
    os.path.join(project_path, "frtb_sa_capital_summary.csv"),
    index=False
)

print("\nCapital component summary exported successfully.")

FRTB-SA Risk Component Summary
  Risk_Component  Calculated_Amount
0          Delta            32450.0
1           Vega             2100.0
2      Curvature                0.0
3            DRC             4875.0
4           RRAO            10000.0

Capital component summary exported successfully.


Step 11 – FRTB-SA Results Validation and Final Reporting

In [20]:
# Calculate total sensitivity amounts
total_delta = delta_report["Delta_Sensitivity"].abs().sum()
total_vega = vega_report["Vega_Sensitivity"].abs().sum()
total_curvature = curvature_report["Curvature_Charge"].sum()

# Calculate additional risk charges
total_drc = drc_report["DRC"].sum()
total_rrao = rrao_report["RRAO"].sum()

# Prepare consolidated risk summary
capital_summary = pd.DataFrame({
    "Risk_Component": ["Delta", "Vega", "Curvature", "DRC", "RRAO"],
    "Risk_Charge": [total_delta, total_vega, total_curvature, total_drc, total_rrao]
})

print(capital_summary)

  Risk_Component  Risk_Charge
0          Delta      32450.0
1           Vega       2100.0
2      Curvature          0.0
3            DRC       4875.0
4           RRAO      10000.0


In [21]:
illustrative_capital = capital_summary["Risk_Charge"].sum()

print("\nIllustrative Total Risk Amount:", round(illustrative_capital, 2))


Illustrative Total Risk Amount: 49425.0


In [22]:
# Check for missing values in the risk reports
print("Delta missing values:", delta_report.isnull().sum().sum())
print("Vega missing values:", vega_report.isnull().sum().sum())
print("Curvature missing values:", curvature_report.isnull().sum().sum())
print("DRC missing values:", drc_report.isnull().sum().sum())
print("RRAO missing values:", rrao_report.isnull().sum().sum())

# Check whether trade IDs are duplicated in the original trade book
print("\nDuplicate Trade IDs:", trade_book["Trade_ID"].duplicated().sum())

# Check whether all risk charges are non-negative
print("Negative risk charges:", (capital_summary["Risk_Charge"] < 0).sum())

Delta missing values: 0
Vega missing values: 0
Curvature missing values: 0
DRC missing values: 0
RRAO missing values: 0

Duplicate Trade IDs: 1
Negative risk charges: 0


In [23]:
# Add the illustrative total to the report
capital_summary.loc[len(capital_summary)] = [
    "Illustrative Total",
    illustrative_capital]

capital_summary.to_csv(
    os.path.join(project_path, "frtb_sa_capital_summary.csv"),
    index=False)

print("\nFRTB-SA capital summary exported successfully.")


FRTB-SA capital summary exported successfully.


Step 12 – Defining Risk Weights and Calculating Risk-Weighted Sensitivities

In [24]:
# Define illustrative Delta risk weights by risk class
delta_risk_weights = {
    "Equity": 0.30,
    "GIRR": 0.01,
    "CSR": 0.05,
    "FX": 0.15,
    "Commodity": 0.20}

# Apply risk weights to Delta sensitivities
delta_weighted_report = delta_report.copy()

delta_weighted_report["Risk_Weight"] = (
    delta_weighted_report["Risk_Class"].map(delta_risk_weights))

delta_weighted_report["Weighted_Sensitivity"] = (
    delta_weighted_report["Delta_Sensitivity"] *
    delta_weighted_report["Risk_Weight"])

# Define illustrative Vega risk weight
vega_risk_weight = 0.25

# Apply risk weight to Vega sensitivities
vega_weighted_report = vega_report.copy()

vega_weighted_report["Risk_Weight"] = vega_risk_weight

vega_weighted_report["Weighted_Sensitivity"] = (
    vega_weighted_report["Vega_Sensitivity"] *
    vega_weighted_report["Risk_Weight"])

# Display weighted Delta sensitivities
print("Delta Risk-Weighted Sensitivities:")
print(delta_weighted_report[
    ["Trade_ID", "Risk_Class", "Delta_Sensitivity",
     "Risk_Weight", "Weighted_Sensitivity"]])

# Display weighted Vega sensitivities
print("\nVega Risk-Weighted Sensitivities:")
print(vega_weighted_report[
    ["Trade_ID", "Risk_Class", "Vega_Sensitivity",
     "Risk_Weight", "Weighted_Sensitivity"]])

# Export reports
delta_weighted_report.to_csv(
    os.path.join(project_path, "delta_weighted_sensitivities.csv"),
    index=False)

vega_weighted_report.to_csv(
    os.path.join(project_path, "vega_weighted_sensitivities.csv"),
    index=False)

print("\nRisk-weighted sensitivity reports exported successfully.")

Delta Risk-Weighted Sensitivities:
   Trade_ID Risk_Class  Delta_Sensitivity  Risk_Weight  Weighted_Sensitivity
0     TR001     Equity             5000.0         0.30                1500.0
1     TR002     Equity             3000.0         0.30                 900.0
2     TR003       GIRR             -850.0         0.01                  -8.5
3     TR004       GIRR             -600.0         0.01                  -6.0
4     TR004        CSR             -600.0         0.05                 -30.0
5     TR005         FX             5000.0         0.15                 750.0
6     TR006  Commodity             4000.0         0.20                 800.0
7     TR007     Equity             1200.0         0.30                 360.0
8     TR008     Equity             -600.0         0.30                -180.0
9     TR009       GIRR            -1200.0         0.01                 -12.0
10    TR010         FX             6000.0         0.15                 900.0
11    TR011  Commodity            -3000.0

Step 13 – Risk Bucket Allocation and Correlation Matrix

In [25]:
# Create an illustrative Equity risk bucket mapping
equity_buckets = pd.DataFrame({
    "Risk_Factor": ["Apple Share Price", "Microsoft Share Price"],
    "Risk_Bucket": ["Apple", "Microsoft"]
})

# Define illustrative correlation matrix
equity_correlation_matrix = pd.DataFrame(
    [[1.00, 0.50],
     [0.50, 1.00]],
    index=["Apple Share Price", "Microsoft Share Price"],
    columns=["Apple Share Price", "Microsoft Share Price"]
)

# Display risk bucket mapping
print("Equity Risk Bucket Mapping:")
print(equity_buckets)

# Display correlation matrix
print("\nEquity Correlation Matrix:")
print(equity_correlation_matrix)

# Export both reports
equity_buckets.to_csv(
    os.path.join(project_path, "equity_risk_buckets.csv"),
    index=False
)

equity_correlation_matrix.to_csv(
    os.path.join(project_path, "equity_correlation_matrix.csv")
)

print("\nRisk bucket and correlation reports exported successfully.")

Equity Risk Bucket Mapping:
             Risk_Factor Risk_Bucket
0      Apple Share Price       Apple
1  Microsoft Share Price   Microsoft

Equity Correlation Matrix:
                       Apple Share Price  Microsoft Share Price
Apple Share Price                    1.0                    0.5
Microsoft Share Price                0.5                    1.0

Risk bucket and correlation reports exported successfully.


Next: Step 14 – Delta, Vega and Curvature Aggregation.

In [26]:
# Select Equity Delta weighted sensitivities
equity_delta = delta_weighted_report[
    delta_weighted_report["Risk_Class"] == "Equity"
].copy()

# Aggregate sensitivities by risk factor
equity_delta = equity_delta.groupby("Risk_Factor")[
    "Weighted_Sensitivity"
].sum()

# Align sensitivities with the correlation matrix
equity_delta = equity_delta.reindex(
    equity_correlation_matrix.index, fill_value=0
)

# Calculate bucket-level Delta capital
delta_vector = equity_delta.values
correlation_matrix = equity_correlation_matrix.values

equity_delta_capital = np.sqrt(max(
    0,
    delta_vector @ correlation_matrix @ delta_vector
))

print("Equity Delta Aggregation:")
print(equity_delta)

print("\nEquity Delta Capital:", round(equity_delta_capital, 2))

Equity Delta Aggregation:
Apple Share Price        1860.0
Microsoft Share Price     720.0
Name: Weighted_Sensitivity, dtype: float64

Equity Delta Capital: 2305.91


In [27]:
# Aggregate Equity Vega sensitivities by risk factor
equity_vega = vega_weighted_report.groupby("Risk_Factor")[
    "Weighted_Sensitivity"
].sum()

# Align with the correlation matrix
equity_vega = equity_vega.reindex(
    equity_correlation_matrix.index, fill_value=0
)

# Calculate bucket-level Vega capital
vega_vector = equity_vega.values

equity_vega_capital = np.sqrt(max(
    0,
    vega_vector @ correlation_matrix @ vega_vector
))

print("Equity Vega Aggregation:")
print(equity_vega)

print("\nEquity Vega Capital:", round(equity_vega_capital, 2))

Equity Vega Aggregation:
Apple Share Price        0.0
Microsoft Share Price    0.0
Name: Weighted_Sensitivity, dtype: float64

Equity Vega Capital: 0.0


In [28]:
# Calculate illustrative curvature charge
equity_curvature_capital = curvature_report["Curvature_Charge"].sum()

print("Equity Curvature Charge:", round(equity_curvature_capital, 2))

Equity Curvature Charge: 0.0


In [29]:
# Prepare illustrative Equity SBM summary
equity_sbm_summary = pd.DataFrame({
    "Risk_Component": ["Delta", "Vega", "Curvature"],
    "Illustrative_Capital": [
        equity_delta_capital,
        equity_vega_capital,
        equity_curvature_capital
    ]
})

print(equity_sbm_summary)

equity_sbm_summary.to_csv(
    os.path.join(project_path, "equity_sbm_aggregation.csv"),
    index=False
)

print("\nEquity SBM aggregation report exported successfully.")

  Risk_Component  Illustrative_Capital
0          Delta           2305.905462
1           Vega              0.000000
2      Curvature              0.000000

Equity SBM aggregation report exported successfully.
